In [2]:
# 1. Clone repository
!rm -rf /kaggle/working/Project-CLARITY
!git clone https://github.com/Git-Dileep/Project-CLARITY.git /kaggle/working/Project-CLARITY

# 2. Enter workstream directory
%cd /kaggle/working/Project-CLARITY/detection/workstream

# 3. Install required packages
!pip install -r requirements.txt

Cloning into '/kaggle/working/Project-CLARITY'...
remote: Enumerating objects: 703, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 703 (delta 103), reused 185 (delta 78), pack-reused 470 (from 1)
Receiving objects: 100% (703/703), 602.75 KiB | 3.79 MiB/s, done.
Resolving deltas: 100% (332/332), done.
/kaggle/working/Project-CLARITY/detection/workstream


In [5]:
import os, sys, json, time, shutil, torch, joblib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

# Set directory paths
base_dir = Path("/kaggle/working/Project-CLARITY/detection/workstream")
os.chdir(base_dir)
if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))

# 1. Output directories
reports_dir = Path("/kaggle/working/outputs/reports")
models_dir = Path("/kaggle/working/outputs/models")
reports_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# 2. Write Kaggle config
kaggle_config = {
  "dataset_name": "ACI-IoT-Network-2023",
  "is_synthetic": False,
  "input_path": "/kaggle/input/datasets/emilynack/aci-iot-network-traffic-dataset-2023/ACI-IoT-2023.csv",
  "target_column": "Label",
  "flow_id_column": "Flow ID",
  "timestamp_column": "Timestamp",
  "device_id_column": None,
  "metadata_columns": ["Source IP", "Destination IP", "Flow ID", "Timestamp", "source_file", "sample_id"],
  "drop_columns": ["Source IP", "Destination IP", "Flow ID", "Timestamp"],
  "numeric_columns": None,
  "categorical_columns": None,
  "max_rows_per_class": 50000,
  "test_size": 0.2,
  "random_seed": 42,
  "chunksize": 250000,
  "traditional": {
    "rf_n_estimators": 100,
    "rf_max_depth": 15,
    "xgb_n_estimators": 100,
    "xgb_max_depth": 6,
    "xgb_learning_rate": 0.1,
    "mlp_hidden_layer_sizes": [64, 32],
    "mlp_max_iter": 50,
    "mlp_early_stopping": True
  },
  "dl": {
    "enabled": True,
    "batch_size": 256,
    "epochs": 10,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "cnn_channels": 32,
    "lstm_hidden": 32,
    "transformer_dim": 32,
    "transformer_heads": 4,
    "early_stopping_patience": 3,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
  },
  "explainability": {
    "top_n_features": 10,
    "background_samples": 50,
    "traditional_explainer": "TreeExplainer",
    "dl_explainer": "GradientExplainer"
  },
  "pilot": {
    "pilot_size": 20,
    "min_classes": 5,
    "stratified": True
  },
  "reports_dir": "/kaggle/working/outputs/reports",
  "models_dir": "/kaggle/working/outputs/models",
  "data_dir": "/kaggle/input/datasets/emilynack/aci-iot-network-traffic-dataset-2023"
}

with open("config/kaggle_aci_2023.json", "w") as f:
    json.dump(kaggle_config, f, indent=2)

# 3. Verified imports
from src.config import WorkstreamConfig
from src.audit_datasets import run_audit
from src.data_loader import load_dataset
from src.preprocessing import prepare_data_and_splits
from src.train_traditional import train_traditional_models
from src.train_dl import train_dl_models
from src.export_pilot import select_stratified_pilot_indices, format_alert_id, build_alert_payload
from src.validate_alerts import load_validator, validate_alert_payload
from src.explainability import explain_traditional_tree, explain_dl_gradient

cfg = WorkstreamConfig.load("config/kaggle_aci_2023.json")

print("\n" + "="*60)
print("[STAGE 1/5] Running Dataset Audit...")
print("="*60)
run_audit(cfg)

print("\n" + "="*60)
print("[STAGE 2/5] Loading & Preprocessing ACI-IoT-2023...")
print("="*60)
df, _ = load_dataset(cfg)

# In-memory sanitization: convert inf, -inf, and extreme floats to NaN
df = df.replace(["Infinity", "-Infinity", "inf", "-inf", np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=[np.number]).columns:
    df.loc[df[col] > 1e300, col] = np.nan
    df.loc[df[col] < -1e300, col] = np.nan

splits = prepare_data_and_splits(df, cfg)

print("\n" + "="*60)
print("[STAGE 3/5] Training Traditional Ensemble (RF, XGB, MLP)...")
print("="*60)
trad_res = train_traditional_models(splits, cfg)

print("\n" + "="*60)
print("[STAGE 4/5] Training Deep Learning Ensemble (CNN, Bi-LSTM, Transformer)...")
print("="*60)
dl_res = train_dl_models(splits, cfg)

print("\n" + "="*60)
print("[STAGE 5/5] Generating Matched Pilot Alerts (Consensus SHAP)...")
print("="*60)
pilot_indices = select_stratified_pilot_indices(splits.test_df, cfg.target_column, cfg.pilot.pilot_size, cfg.random_seed)
pilot_test_df = splits.test_df.loc[pilot_indices].copy().reset_index(drop=True)
X_pilot = splits.X_test[pilot_indices]
y_pilot_true = [splits.class_names[i] for i in splits.y_test[pilot_indices]]
flow_ids = [str(pilot_test_df.iloc[i].get("Flow ID", f"FLOW-{i:06d}")) for i in range(len(pilot_test_df))]
now_dt = datetime.now(timezone.utc)
timestamps = [now_dt.isoformat()] * len(pilot_test_df)

# Traditional predictions & SHAP
rf = trad_res["models"]["rf"]
xgb = trad_res["models"]["xgb"]
mlp = trad_res["models"]["mlp"]
rf_p = rf.predict_proba(X_pilot)
xgb_p = xgb.predict_proba(X_pilot)
mlp_p = mlp.predict_proba(X_pilot)

trad_preds, trad_confs, trad_votes = [], [], []
for i in range(len(pilot_indices)):
    v_r = splits.class_names[np.argmax(rf_p[i])]
    v_x = splits.class_names[np.argmax(xgb_p[i])]
    v_m = splits.class_names[np.argmax(mlp_p[i])]
    maj = Counter([v_r, v_x, v_m]).most_common(1)[0][0]
    trad_preds.append(maj)
    trad_confs.append(float(np.mean([np.max(rf_p[i]), np.max(xgb_p[i]), np.max(mlp_p[i])])))
    trad_votes.append([
        {"learner_name": "RandomForest", "voted_label": v_r, "learner_confidence": float(np.max(rf_p[i]))},
        {"learner_name": "XGBoost", "voted_label": v_x, "learner_confidence": float(np.max(xgb_p[i]))},
        {"learner_name": "MLP", "voted_label": v_m, "learner_confidence": float(np.max(mlp_p[i]))}
    ])

trad_shaps = explain_traditional_tree(
    model=xgb,
    X_samples=X_pilot,
    feature_names=splits.feature_names,
    class_names=splits.class_names,
    predicted_classes=trad_preds
)

# DL predictions & SHAP (evaluated on CPU to prevent device mismatch)
cnn = dl_res["models"]["CNN1D"].eval()
lstm = dl_res["models"]["LSTM"].eval()
transformer = dl_res["models"]["Transformer"].eval()
device = next(transformer.parameters()).device

with torch.no_grad():
    x_t = torch.tensor(X_pilot, dtype=torch.float32).to(device)
    c_p = torch.softmax(cnn(x_t), dim=1).cpu().numpy()
    l_p = torch.softmax(lstm(x_t), dim=1).cpu().numpy()
    t_p = torch.softmax(transformer(x_t), dim=1).cpu().numpy()

dl_preds, dl_confs, dl_votes = [], [], []
for i in range(len(pilot_indices)):
    v_c = splits.class_names[np.argmax(c_p[i])]
    v_l = splits.class_names[np.argmax(l_p[i])]
    v_t = splits.class_names[np.argmax(t_p[i])]
    maj = Counter([v_c, v_l, v_t]).most_common(1)[0][0]
    dl_preds.append(maj)
    dl_confs.append(float(np.mean([np.max(c_p[i]), np.max(l_p[i]), np.max(t_p[i])])))
    dl_votes.append([
        {"learner_name": "CNN1D", "voted_label": v_c, "learner_confidence": float(np.max(c_p[i]))},
        {"learner_name": "LSTM", "voted_label": v_l, "learner_confidence": float(np.max(l_p[i]))},
        {"learner_name": "Transformer", "voted_label": v_t, "learner_confidence": float(np.max(t_p[i]))}
    ])

# Move model to CPU for GradientExplainer
transformer.to("cpu")
bg_samples = splits.X_train[:min(cfg.explainability.background_samples, len(splits.X_train))]
dl_shaps = explain_dl_gradient(
    model=transformer,
    background_samples=bg_samples,
    X_samples=X_pilot,
    feature_names=splits.feature_names,
    class_names=splits.class_names,
    predicted_classes=dl_preds,
    device=torch.device("cpu")
)

# Schema validation
contracts_path = Path("/kaggle/working/Project-CLARITY/contracts/alert_schema_v1.json")
validator = load_validator(contracts_path)

trad_alerts, dl_alerts = [], []
for i in range(len(pilot_test_df)):
    t_payload = build_alert_payload(
        alert_id=format_alert_id(i, now_dt),
        flow_id=flow_ids[i],
        detector_family="traditional_ml",
        timestamp=timestamps[i],
        dataset=cfg.dataset_name,
        predicted_label=trad_preds[i],
        confidence=trad_confs[i],
        base_learner_votes=trad_votes[i],
        shap_attribution=trad_shaps[i],
        true_label=y_pilot_true[i],
        in_train_leakage_flag=False,
    )
    trad_alerts.append(t_payload)

    d_payload = build_alert_payload(
        alert_id=format_alert_id(i + 100, now_dt),
        flow_id=flow_ids[i],
        detector_family="dl_ensemble",
        timestamp=timestamps[i],
        dataset=cfg.dataset_name,
        predicted_label=dl_preds[i],
        confidence=dl_confs[i],
        base_learner_votes=dl_votes[i],
        shap_attribution=dl_shaps[i],
        true_label=y_pilot_true[i],
        in_train_leakage_flag=False,
    )
    dl_alerts.append(d_payload)

with open(reports_dir / "pilot_traditional_alerts.json", "w") as f:
    json.dump(trad_alerts, f, indent=2)
with open(reports_dir / "pilot_dl_alerts.json", "w") as f:
    json.dump(dl_alerts, f, indent=2)

print("\n" + "="*60)
print("CREATING ZIP ARCHIVE FOR DOWNLOAD...")
print("="*60)
shutil.make_archive("/kaggle/working/clarity_real_aci_results", "zip", "/kaggle/working/outputs")

print("\n🎉 SUCCESS! PIPELINE COMPLETED IN FULL.")
print(f"Traditional Alerts: {len(trad_alerts)} | DL Alerts: {len(dl_alerts)}")
print("Zip archive created at: /kaggle/working/clarity_real_aci_results.zip")


[STAGE 1/5] Running Dataset Audit...
[AUDIT] Inspecting ACI-IoT-2023.csv...
[AUDIT] Audit finished successfully. Reports generated in /kaggle/working/outputs/reports

[STAGE 2/5] Loading & Preprocessing ACI-IoT-2023...
[DATA_LOADER] Loading ACI-IoT-2023.csv...
[PREPROCESSING] Numeric features (79): ['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length'

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(



[TRADITIONAL] Ensemble Training Completed Successfully!
  Ensemble Accuracy: 0.9937
  Ensemble Macro F1: 0.9075
  Ensemble Weighted F1: 0.9937
  Reports saved to /kaggle/working/outputs/reports

[STAGE 4/5] Training Deep Learning Ensemble (CNN, Bi-LSTM, Transformer)...

[DEEP LEARNING] PyTorch Version: 2.11.0+cu130 | Executing on: cuda

[DEEP LEARNING] Training CNN1D...
  [CNN1D] Epoch 01/10 - Train Loss: 2.1301 - Val Loss: 1.9425
  [CNN1D] Epoch 02/10 - Train Loss: 1.7112 - Val Loss: 1.6228
  [CNN1D] Epoch 03/10 - Train Loss: 1.5706 - Val Loss: 3.2740
  [CNN1D] Epoch 04/10 - Train Loss: 1.5034 - Val Loss: 1.5762
  [CNN1D] Epoch 05/10 - Train Loss: 1.4448 - Val Loss: 1.4310
  [CNN1D] Epoch 06/10 - Train Loss: 1.3924 - Val Loss: 9.3241
  [CNN1D] Epoch 07/10 - Train Loss: 1.3514 - Val Loss: 15.0372
  [CNN1D] Epoch 08/10 - Train Loss: 1.3202 - Val Loss: 4.0721
  [CNN1D] Early stopping triggered at epoch 8.

[DEEP LEARNING] Training LSTM...
  [LSTM] Epoch 01/10 - Train Loss: 1.1809 - Val 